In [61]:
import jams
import glob

In [62]:
# billboard 0-889

In [63]:
HARTE_TO_ABC = {
    "maj":     "",
    "min":     "m",
    "dim":     "dim",
    "aug":     "+",
    "maj7":    "maj7",
    "min7":    "m7",
    "7":       "7",
    "dim7":    "dim7",
    "hdim7":   "m7b5",
    "minmaj7": "mM7",
    "maj6":    "6",
    "min6":    "m6",
    "9":       "9",
    "maj9":    "maj9",
    "min9":    "m9",
    "sus2":    "sus2",
    "sus4":    "sus4",
    "7sus4":   "7sus4",
    "aug7":    "aug7",
    "5":       "5",
    "add9":    "add9",
    "add11":   "add11",
    "6/9":     "6/9",
    "11":      "11",
    "min11":   "m11",
    "maj11":   "maj11",
    "13":      "13",
    "min13":   "m13",
    "maj13":   "maj13",
    "7b9":     "7b9",
    "7#9":     "7#9",
    "7#11":    "7#11",
    "7b13":    "7b13",
    "1":       "",
    "sus4(b7)": "7sus4"
}
 
# Chromatic note names for bass resolution
CHROMATIC = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
 
ROOT_TO_SEMITONE = {
    "C": 0,  "C#": 1, "Db": 1,  "D": 2,  "D#": 3,  "Eb": 3,
    "E": 4,  "F":  5, "F#": 6,  "Gb": 6, "G":  7,  "G#": 8,
    "Ab": 8, "A":  9, "A#": 10, "Bb": 10, "B": 11,
}
 
DEGREE_TO_SEMITONE = {
    "1": 0,  "2": 2,  "b2": 1,  "#2": 3,
    "3": 4,  "b3": 3, "#3": 5,
    "4": 5,  "#4": 6,
    "5": 7,  "b5": 6, "#5": 8,
    "6": 9,  "b6": 8, "#6": 10,
    "7": 11, "b7": 10, "#7": 12,
}


In [64]:
def resolve_bass(root: str, degree: str) -> str:
    """Resolve a scale-degree bass to a note name (e.g. root='F', degree='5' -> 'C')."""
    root_semi = ROOT_TO_SEMITONE.get(root, 0)
    offset = DEGREE_TO_SEMITONE.get(degree)
    if offset is None:
        return degree  # Unknown degree — pass through as-is
    return CHROMATIC[(root_semi + offset) % 12]

In [65]:
def harte_to_abc(label: str) -> str:
    """
    Convert a Harte chord label to an ABC quoted chord symbol string.
 
    Examples:
        "C:maj"    -> '"C"'
        "D:min7"   -> '"Dm7"'
        "F:maj/5"  -> '"F/C"'
        "F:maj/2"  -> '"F/A"'
        "N"        -> '"N.C."'
    """
    if not label or label in ("N", "X"):
        return '"N.C."'
 
    bass_suffix = ""
    if "/" in label:
        label, degree = label.rsplit("/", 1)
        root_temp = label.split(":")[0] if ":" in label else label
        bass_suffix = "/" + resolve_bass(root_temp, degree)
 
    if ":" in label:
        root, shorthand = label.split(":", 1)
    else:
        root, shorthand = label, "maj"
    
    shorthand = shorthand.replace("(", "").replace(")", "")
        
    if shorthand not in HARTE_TO_ABC:
        raise ValueError(f"Unknown Harte shorthand: '{shorthand}' (from '{label}')")
 
    quality = HARTE_TO_ABC.get(shorthand, shorthand)
    return f'"{root}{quality}{bass_suffix}"'

In [66]:
for jam_path in glob.glob("v1.0.0/v1.0.0/jams/billboard_*.jams"):
    print("Processing:", jam_path)
    
    # Load without validation (avoids schema errors)
    jam = jams.load(jam_path, validate=False)
    ann = jam["annotations"][0]["data"]

    # Clean annotations: drop invalid points and enforce monotonic time
    clean_ann = []
    prev_time = -1.0
    for obs in ann:
        if obs.time < 0 or obs.duration < 0:
            continue
        if obs.time < prev_time:
            continue  # skip non-monotonic time
        prev_time = obs.time
        clean_ann.append(obs)

    # Convert to ABC after cleaning
    chords = [(obs.time, harte_to_abc(obs.value)) for obs in clean_ann]

    print(jam_path, chords)

Processing: v1.0.0/v1.0.0/jams\billboard_0.jams
v1.0.0/v1.0.0/jams\billboard_0.jams [(0.0, '"N.C."'), (0.208979591, '"N.C."'), (1.1283125464166666, '"N.C."'), (1.4347568648888889, '"C"'), (3.8863114126666662, '"F"'), (6.337865960444448, '"C"'), (8.789420508222232, '"F"'), (10.934530737527798, '"F"'), (11.240975056, '"C"'), (13.095273525437502, '"Gm"'), (13.713373015250005, '"F"'), (16.185770974500006, '"C"'), (18.040069443937508, '"Gm"'), (18.65816893375001, '"F"'), (21.130566893, '"C"'), (22.968691892937493, '"Gm"'), (23.58140022624999, '"F"'), (25.41952522618748, '"Em"'), (26.03223355949998, '"Dm"'), (27.257650226124973, '"G"'), (28.483066892749967, '"C"'), (30.933900226, '"C"'), (32.759015021874994, '"Gm"'), (33.36738662049999, '"F"'), (35.800873014999986, '"C"'), (37.62598781087498, '"Gm"'), (38.23435940949998, '"F"'), (40.667845804, '"C"'), (42.46266298093749, '"Gm"'), (43.06093537324998, '"F"'), (44.85575255018747, '"Em"'), (45.45402494249996, '"Dm"'), (46.65056972712495, '"G"'),

ValueError: Unknown Harte shorthand: 'sus4b7' (from 'E:sus4(b7)')